In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded!')

✅ Libraries loaded!


In [5]:
results = pd.read_csv('results.csv')

print(f'Shape: {results.shape}')
print(f'Columns: {list(results.columns)}')
print(f'\nDate range: {results["date"].min()} to {results["date"].max()}')

Shape: (49498, 9)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Date range: 1872-11-30 to 2026-07-06


In [6]:
tournament_counts = results['tournament'].value_counts()
print(tournament_counts.head(20))

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1057
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


In [7]:
results.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [8]:
# Filter to post-1990 and remove friendlies
results['date'] = pd.to_datetime(results['date'])

filtered = results[
    (results['date'].dt.year >= 1990) &
    (results['tournament'] != 'Friendly')
].copy()

print(f'Original dataset: {len(results)} matches')
print(f'Filtered dataset: {len(filtered)} matches')
print(f'Removed: {len(results) - len(filtered)} matches')
print(f'\nDate range: {filtered["date"].min()} to {filtered["date"].max()}')

Original dataset: 49498 matches
Filtered dataset: 21551 matches
Removed: 27947 matches

Date range: 1990-01-29 00:00:00 to 2026-07-06 00:00:00


In [9]:
# What tournaments remain after filtering?
print(filtered['tournament'].value_counts().head(15))


tournament
FIFA World Cup qualification            6977
UEFA Euro qualification                 2114
African Cup of Nations qualification    1865
UEFA Nations League                      658
FIFA World Cup                           645
African Cup of Nations                   642
AFC Asian Cup qualification              632
CFU Caribbean Cup qualification          508
CECAFA Cup                               422
CONCACAF Nations League                  422
Gold Cup                                 420
Island Games                             384
Copa América                             378
COSAFA Cup                               354
UEFA Euro                                323
Name: count, dtype: int64


In [10]:
# Tournaments to drop - low quality or irrelevant competitions
drop_tournaments = [
    'Island Games',
    'CECAFA Cup', 
    'COSAFA Cup',
    'CFU Caribbean Cup qualification',
]

filtered = filtered[~filtered['tournament'].isin(drop_tournaments)].copy()

print(f'Matches after dropping low quality tournaments: {len(filtered)}')
print(f'\nRemaining tournaments:')
print(filtered['tournament'].value_counts().head(20))

Matches after dropping low quality tournaments: 19883

Remaining tournaments:
tournament
FIFA World Cup qualification            6977
UEFA Euro qualification                 2114
African Cup of Nations qualification    1865
UEFA Nations League                      658
FIFA World Cup                           645
African Cup of Nations                   642
AFC Asian Cup qualification              632
CONCACAF Nations League                  422
Gold Cup                                 420
Copa América                             378
UEFA Euro                                323
AFC Asian Cup                            298
AFF Championship                         291
Gulf Cup                                 259
CFU Caribbean Cup                        210
UNCAF Cup                                164
SAFF Cup                                 162
Confederations Cup                       140
EAFF Championship                        130
Asian Games                              120
Name: count

In [11]:
# Create outcome column from home team's perspective
def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 'W'
    elif row['home_score'] < row['away_score']:
        return 'L'
    else:
        return 'D'

filtered['outcome'] = filtered.apply(get_outcome, axis=1)

# Check the distribution
print(filtered['outcome'].value_counts())
print(f'\nAs percentages:')
print(filtered['outcome'].value_counts(normalize=True).round(3) * 100)

outcome
W    9731
L    5821
D    4331
Name: count, dtype: int64

As percentages:
outcome
W    48.9
L    29.3
D    21.8
Name: proportion, dtype: float64


In [12]:
##  Home Field Advantage Analysis

### Outcome Distribution
- **Win (Home): 48.9%** 
- **Loss (Home): 29.3%**
- **Draw: 21.8%**

Home teams win nearly half of all matches, significantly more 
than away teams at 29.3%. This is a well known phenomenon in 
football called **home field advantage**.

### Why Home Teams Win More
1. **Crowd support** — home fans create pressure on the 
   opposition and can influence referee decisions subconsciously
2. **Travel fatigue** — away teams travel long distances, 
   disrupting sleep, nutrition and preparation
3. **Familiarity** — home teams know the pitch conditions, 
   climate and altitude
4. **Referee bias** — studies show referees award more injury 
   time and are more lenient toward the home side due to 
   crowd pressure

### Implication for Our Model
The `neutral` column in our dataset flags matches played at 
neutral venues. At neutral venues, home advantage should 
disappear — making it an important feature for predicting 
World Cup matches, which are hosted in a single country.

SyntaxError: invalid character '—' (U+2014) (2367136690.py, line 13)

In [13]:
# Does neutral venue affect home advantage?
print('=== Home Win % by Venue Type ===')
print(filtered.groupby('neutral')['outcome'].value_counts(normalize=True).round(3) * 100)

=== Home Win % by Venue Type ===
neutral  outcome
False    W          51.5
         L          27.2
         D          21.3
True     W          43.1
         L          34.1
         D          22.8
Name: proportion, dtype: float64


In [14]:
## 📊 Neutral Venue Confirmation

Data confirms home advantage disappears at neutral venues:

| Venue Type | Home Win% | Away Win% | Gap |
|---|---|---|---|
| Home Ground | 51.5% | 27.2% | 24.3% |
| Neutral | 43.1% | 34.1% | 9.0% |

This validates including `neutral` as a feature in our model.
World Cup matches are mostly neutral — so home advantage is 
significantly reduced compared to regular international football.

SyntaxError: invalid character '—' (U+2014) (3696312757.py, line 11)

In [15]:
# Sort by date - crucial for calculating historical features correctly
filtered = filtered.sort_values('date').reset_index(drop=True)

print(f'Date range confirmed: {filtered["date"].min()} to {filtered["date"].max()}')
print(f'\nFirst 3 matches:')
print(filtered[['date', 'home_team', 'away_team', 'home_score', 'away_score']].head(3))
print(f'\nLast 3 matches:')
print(filtered[['date', 'home_team', 'away_team', 'home_score', 'away_score']].tail(3))

Date range confirmed: 1990-01-29 00:00:00 to 2026-07-06 00:00:00

First 3 matches:
        date  home_team away_team  home_score  away_score
0 1990-01-29   Thailand     Kenya         2.0         1.0
1 1990-02-02   Colombia   Uruguay         0.0         2.0
2 1990-02-02  Indonesia     Kenya         2.0         3.0

Last 3 matches:
            date      home_team away_team  home_score  away_score
19880 2026-07-05         Mexico   England         NaN         NaN
19881 2026-07-05         Brazil    Norway         NaN         NaN
19882 2026-07-06  United States   Belgium         NaN         NaN


In [16]:
def get_team_stats(team, date, df, n=10):
    """
    Calculate team statistics from their last N matches
    before a given date. Only looks backwards - no data leakage.
    
    Returns: dict of features for that team at that point in time
    """
    # Get all matches this team played BEFORE this date
    team_matches = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    ].tail(n)  # Take only last N matches
    
    # If team has no history return neutral stats
    if len(team_matches) == 0:
        return {
            'games_played': 0,
            'win_rate': 0.5,
            'draw_rate': 0.2,
            'avg_goals_scored': 1.0,
            'avg_goals_conceded': 1.0,
        }
    
    wins, draws, goals_scored, goals_conceded = 0, 0, 0, 0
    
    for _, match in team_matches.iterrows():
        if match['home_team'] == team:
            gf = match['home_score']  # Goals for
            ga = match['away_score']  # Goals against
        else:
            gf = match['away_score']
            ga = match['home_score']
        
        if gf > ga:
            wins += 1
        elif gf == ga:
            draws += 1
            
        goals_scored += gf
        goals_conceded += ga
    
    n_matches = len(team_matches)
    
    return {
        'games_played': n_matches,
        'win_rate': wins / n_matches,
        'draw_rate': draws / n_matches,
        'avg_goals_scored': goals_scored / n_matches,
        'avg_goals_conceded': goals_conceded / n_matches,
    }

print('✅ Team stats function defined!')

✅ Team stats function defined!


In [17]:
# Test our function on a real team
brazil_stats = get_team_stats(
    team='Brazil',
    date=pd.Timestamp('2026-07-01'),
    df=filtered
)

print('=== Brazil stats before July 2026 ===')
for key, value in brazil_stats.items():
    print(f'{key}: {value:.3f}')

=== Brazil stats before July 2026 ===
games_played: 10.000
win_rate: 0.500
draw_rate: 0.200
avg_goals_scored: 1.600
avg_goals_conceded: 1.000


In [18]:
# Test our function on a real team
argentina_stats = get_team_stats(
    team='Argentina',
    date=pd.Timestamp('2026-07-01'),
    df=filtered
)

print('=== Argentina stats before July 2026 ===')
for key, value in argentina_stats.items():
    print(f'{key}: {value:.3f}')

=== Argentina stats before July 2026 ===
games_played: 10.000
win_rate: 0.800
draw_rate: 0.100
avg_goals_scored: 1.900
avg_goals_conceded: 0.400


In [19]:
# Apply team stats to every match - this builds our feature matrix
# Warning: this may take a minute or two to run

print('Building features for all matches...')
print('This calculates team stats for each of our 19,883 matches')
print('Please wait...\n')

home_stats = []
away_stats = []

for idx, row in filtered.iterrows():
    h = get_team_stats(row['home_team'], row['date'], filtered)
    a = get_team_stats(row['away_team'], row['date'], filtered)
    home_stats.append(h)
    away_stats.append(a)

# Convert to dataframes
home_df = pd.DataFrame(home_stats).add_prefix('home_')
away_df = pd.DataFrame(away_stats).add_prefix('away_')

print('✅ Features built!')
print(f'Home features shape: {home_df.shape}')
print(f'Away features shape: {away_df.shape}')

Building features for all matches...
This calculates team stats for each of our 19,883 matches
Please wait...

✅ Features built!
Home features shape: (19883, 5)
Away features shape: (19883, 5)


In [20]:
# Combine all features into one dataframe
model_df = pd.concat([
    filtered[['date', 'home_team', 'away_team', 
              'tournament', 'neutral', 'outcome']].reset_index(drop=True),
    home_df.reset_index(drop=True),
    away_df.reset_index(drop=True)
], axis=1)

# Drop rows where we have NaN scores (upcoming matches)
model_df = model_df.dropna()

print(f'Final dataset shape: {model_df.shape}')
print(f'\nColumns: {list(model_df.columns)}')
print(f'\nSample row:')
model_df.tail(3)

Final dataset shape: (19883, 16)

Columns: ['date', 'home_team', 'away_team', 'tournament', 'neutral', 'outcome', 'home_games_played', 'home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'away_games_played', 'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded']

Sample row:


,date,home_team,away_team,tournament,neutral,outcome,home_games_played,home_win_rate,home_draw_rate,home_avg_goals_scored,home_avg_goals_conceded,away_games_played,away_win_rate,away_draw_rate,away_avg_goals_scored,away_avg_goals_conceded
19880,2026-07-05,Mexico,England,FIFA World Cup,False,D,10,0.9,0.1,1.8,0.3,10,0.9,0.1,2.5,0.3
19881,2026-07-05,Brazil,Norway,FIFA World Cup,True,D,10,0.5,0.2,1.6,1.0,10,0.9,0.0,3.8,1.1
19882,2026-07-06,United States,Belgium,FIFA World Cup,False,D,10,0.7,0.1,2.3,1.0,10,0.6,0.4,3.3,0.7


In [21]:
# Check these specific matches in the original filtered data
mask = filtered['date'] >= '2026-07-05'
print(filtered[mask][['date', 'home_team', 'away_team', 
                       'home_score', 'away_score']].head(10))

            date      home_team away_team  home_score  away_score
19880 2026-07-05         Mexico   England         NaN         NaN
19881 2026-07-05         Brazil    Norway         NaN         NaN
19882 2026-07-06  United States   Belgium         NaN         NaN


In [22]:
# Fix - drop matches where scores were NaN in original data
valid_mask = filtered['home_score'].notna() & filtered['away_score'].notna()
model_df = model_df[valid_mask.values]

print(f'Shape after removing matches with no scores: {model_df.shape}')
print(f'\nLatest matches now:')
model_df.tail(3)[['date', 'home_team', 'away_team', 'outcome']]

Shape after removing matches with no scores: (19872, 16)

Latest matches now:


,date,home_team,away_team,outcome
19869,2026-07-01,England,DR Congo,W
19870,2026-07-01,Belgium,Senegal,W
19871,2026-07-01,United States,Bosnia and Herzegovina,W


In [23]:
# Define features and target
drop_cols = ['date', 'home_team', 'away_team', 'tournament', 'outcome']

X = model_df.drop(columns=drop_cols)
y = model_df['outcome']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeatures we are using:')
print(list(X.columns))
print(f'\nTarget classes: {y.unique()}')

Features shape: (19872, 11)
Target shape: (19872,)

Features we are using:
['neutral', 'home_games_played', 'home_win_rate', 'home_draw_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'away_games_played', 'away_win_rate', 'away_draw_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded']

Target classes: <StringArray>
['W', 'L', 'D']
Length: 3, dtype: str


In [24]:
# Time based train/test split
split_date = pd.Timestamp('2022-01-01')

train_mask = model_df['date'] < split_date
test_mask = model_df['date'] >= split_date

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print(f'Training set: {X_train.shape} ({train_mask.sum()} matches)')
print(f'Test set: {X_test.shape} ({test_mask.sum()} matches)')
print(f'\nTraining period: 1990 - 2021')
print(f'Test period: 2022 - 2026')

Training set: (16584, 11) (16584 matches)
Test set: (3288, 11) (3288 matches)

Training period: 1990 - 2021
Test period: 2022 - 2026


In [25]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Encode target (W/D/L → numbers)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f'Classes: {le.classes_}')
print(f'Encoded as: {list(range(len(le.classes_)))}')
print('✅ Data ready for modelling!')

Classes: ['D' 'L' 'W']
Encoded as: [0, 1, 2]
✅ Data ready for modelling!


In [26]:
# Train multiple classification models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results = {}

for name, model in models.items():
    # Use scaled data for logistic regression, raw for tree based
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train_encoded)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train_encoded)
        preds = model.predict(X_test)
    
    accuracy = accuracy_score(y_test_encoded, preds)
    results[name] = {
        'accuracy': round(accuracy * 100, 2),
        'predictions': preds
    }
    print(f'{name:<25} Accuracy: {accuracy*100:.2f}%')

print('\n✅ All models trained!')

Logistic Regression       Accuracy: 55.75%
Random Forest             Accuracy: 53.95%
Gradient Boosting         Accuracy: 56.08%

✅ All models trained!


In [27]:
# Detailed performance breakdown
from sklearn.metrics import classification_report

best_preds = results['Gradient Boosting']['predictions']

print('=== Gradient Boosting - Detailed Report ===\n')
print(classification_report(
    y_test_encoded, 
    best_preds,
    target_names=['Draw', 'Loss', 'Win']
))

=== Gradient Boosting - Detailed Report ===

              precision    recall  f1-score   support

        Draw       0.30      0.01      0.02       736
        Loss       0.55      0.52      0.54      1015
         Win       0.57      0.85      0.68      1537

    accuracy                           0.56      3288
   macro avg       0.47      0.46      0.41      3288
weighted avg       0.50      0.56      0.49      3288



In [28]:
## 🤖 Model Results

| Model | Accuracy |
|---|---|
| Gradient Boosting | 56.08% |
| Logistic Regression | 55.75% |
| Random Forest | 53.95% |

### Key Findings
- 56% accuracy is strong for football prediction 
  (random guessing = 33%)
- Draws are hardest to predict (F1: 0.02) — 
  a known challenge in football analytics
- Model is biased toward predicting wins (most common outcome)
- Professional betting models typically achieve 55-65%, 
  so our baseline is competitive

### Why Draws Are Hard
Draws occur between evenly matched teams where small 
random events decide the game. More features capturing 
team similarity could improve draw prediction.

SyntaxError: invalid character '—' (U+2014) (694251219.py, line 12)

In [29]:
# Improved model with balanced class weights
gb_balanced = GradientBoostingClassifier(
    n_estimators=200, 
    random_state=42
)

# Calculate class weights manually
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight('balanced', y_train_encoded)

# Train with sample weights
gb_balanced.fit(X_train, y_train_encoded, sample_weight=sample_weights)
preds_balanced = gb_balanced.predict(X_test)

print(f'Balanced accuracy: {accuracy_score(y_test_encoded, preds_balanced)*100:.2f}%')
print(f'\n=== Detailed Report ===\n')
print(classification_report(
    y_test_encoded,
    preds_balanced,
    target_names=['Draw', 'Loss', 'Win']
))

Balanced accuracy: 50.21%

=== Detailed Report ===

              precision    recall  f1-score   support

        Draw       0.27      0.35      0.30       736
        Loss       0.53      0.57      0.55      1015
         Win       0.66      0.53      0.59      1537

    accuracy                           0.50      3288
   macro avg       0.49      0.48      0.48      3288
weighted avg       0.53      0.50      0.51      3288



In [30]:
## ⚠️ Important Note on Draws

The World Cup has two distinct phases:
- **Group Stage** — draws are possible (use balanced model)
- **Knockout Stage** — no draws, extra time/penalties decide 
  (use unbalanced model, remove Draw predictions)

### Future Improvement
Train two separate models:
1. Group stage classifier (W/D/L)
2. Knockout classifier (W/L only)

This is how professional football prediction systems work.

SyntaxError: invalid character '—' (U+2014) (2579180981.py, line 4)

In [33]:
# Find upcoming matches (no scores yet)
upcoming = filtered[filtered['home_score'].isna()].copy()

print(f'Upcoming matches: {len(upcoming)}')
print(upcoming[['date', 'home_team', 'away_team', 'tournament']].to_string())

Upcoming matches: 11
            date      home_team   away_team      tournament
19872 2026-07-02          Spain     Austria  FIFA World Cup
19873 2026-07-02       Portugal     Croatia  FIFA World Cup
19874 2026-07-02    Switzerland     Algeria  FIFA World Cup
19875 2026-07-03      Australia       Egypt  FIFA World Cup
19876 2026-07-03      Argentina  Cape Verde  FIFA World Cup
19877 2026-07-03       Colombia       Ghana  FIFA World Cup
19878 2026-07-04         Canada     Morocco  FIFA World Cup
19879 2026-07-04       Paraguay      France  FIFA World Cup
19880 2026-07-05         Mexico     England  FIFA World Cup
19881 2026-07-05         Brazil      Norway  FIFA World Cup
19882 2026-07-06  United States     Belgium  FIFA World Cup


In [34]:
# Predict Argentina vs Cape Verde
def predict_match(home_team, away_team, date, df, model, scaler, le, neutral=True):
    """
    Predict outcome of a match given two teams
    """
    match_date = pd.Timestamp(date)
    
    # Get team stats
    home_stats = get_team_stats(home_team, match_date, df)
    away_stats = get_team_stats(away_team, match_date, df)
    
    # Build feature row
    features = {
        'neutral': 1 if neutral else 0,
        'home_games_played': home_stats['games_played'],
        'home_win_rate': home_stats['win_rate'],
        'home_draw_rate': home_stats['draw_rate'],
        'home_avg_goals_scored': home_stats['avg_goals_scored'],
        'home_avg_goals_conceded': home_stats['avg_goals_conceded'],
        'away_games_played': away_stats['games_played'],
        'away_win_rate': away_stats['win_rate'],
        'away_draw_rate': away_stats['draw_rate'],
        'away_avg_goals_scored': away_stats['avg_goals_scored'],
        'away_avg_goals_conceded': away_stats['avg_goals_conceded'],
    }
    
    feature_df = pd.DataFrame([features])
    prediction = model.predict(feature_df)
    probability = model.predict_proba(feature_df)
    
    outcome = le.inverse_transform(prediction)[0]
    probs = dict(zip(le.classes_, probability[0].round(3)))
    
    return outcome, probs

# Predict Argentina vs Cape Verde
outcome, probs = predict_match(
    home_team='Argentina',
    away_team='Cape Verde',
    date='2026-07-03',
    df=filtered,
    model=models['Gradient Boosting'],
    scaler=scaler,
    le=le,
    neutral=True
)

print(f'=== Argentina vs Cape Verde ===')
print(f'Predicted outcome: {outcome}')
print(f'\nProbabilities:')
for result, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 30)
    print(f'{result}: {bar} {prob*100:.1f}%')

=== Argentina vs Cape Verde ===
Predicted outcome: D

Probabilities:
D: █████████████████ 59.1%
W: █████████ 30.2%
L: ███ 10.7%


In [35]:
# Check Cape Verde's stats
cape_verde_stats = get_team_stats(
    team='Cape Verde',
    date=pd.Timestamp('2026-07-03'),
    df=filtered
)

print('=== Cape Verde stats ===')
for key, value in cape_verde_stats.items():
    print(f'{key}: {value:.3f}')

=== Cape Verde stats ===
games_played: 10.000
win_rate: 0.200
draw_rate: 0.700
avg_goals_scored: 1.300
avg_goals_conceded: 1.100


In [36]:
## ⚠️ Model Limitation - Quality of Opposition

Current model doesn't account for strength of opposition.
Cape Verde's 70% draw rate inflates their perceived strength
since those draws were against weaker African nations.

### Future Improvements
- Add FIFA ranking difference as a feature
- Add ELO rating (accounts for quality of opposition)
- Add head-to-head historical record
- Weight results by opponent strength

ELO ratings are the gold standard in football prediction
and would significantly improve our model.

SyntaxError: unterminated string literal (detected at line 3) (1581249843.py, line 3)

In [37]:
# Predict Australia vs Egypt
outcome, probs = predict_match(
    home_team='Australia',
    away_team='Egypt',
    date='2026-07-03',
    df=filtered,
    model=models['Gradient Boosting'],
    scaler=scaler,
    le=le,
    neutral=True
)

print(f'=== Australia vs Egypt ===')
print(f'Predicted outcome: {outcome}')
print(f'\nProbabilities:')
for result, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 30)
    print(f'{result}: {bar} {prob*100:.1f}%')

=== Australia vs Egypt ===
Predicted outcome: W

Probabilities:
W: ████████████ 41.9%
D: ███████████ 39.0%
L: █████ 19.1%


In [38]:
## 🔍 Prediction Validation - Australia vs Egypt

**Model prediction:** Win (41.9%) — Draw (39.0%) — Loss (19.1%)
**Actual result:** Draw 1-1 (Australia lost 4-2 on penalties)

### Analysis
- Model correctly identified this as a close matchup
- Draw was second most likely at 39% — close to actual outcome
- Penalty shootouts are beyond model scope (essentially random)
- Validates our model captures match competitiveness well

### Limitation
Model predicts 90 minute outcomes only.
Knockout stage requires separate penalty prediction logic.

SyntaxError: invalid character '—' (U+2014) (3559917397.py, line 3)

In [39]:
# Predict Brazil vs Norway
outcome, probs = predict_match(
    home_team='Brazil',
    away_team='Norway',
    date='2026-07-03',
    df=filtered,
    model=models['Gradient Boosting'],
    scaler=scaler,
    le=le,
    neutral=True
)

print(f'=== Brazil vs Norway ===')
print(f'Predicted outcome: {outcome}')
print(f'\nProbabilities:')
for result, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 30)
    print(f'{result}: {bar} {prob*100:.1f}%')

=== Brazil vs Norway ===
Predicted outcome: L

Probabilities:
L: ███████████████ 53.0%
W: ██████████ 35.7%
D: ███ 11.3%


In [40]:
# Check both teams stats
for team, date in [('Brazil', '2026-07-05'), ('Norway', '2026-07-05')]:
    stats = get_team_stats(team, pd.Timestamp(date), filtered)
    print(f'\n=== {team} ===')
    for key, value in stats.items():
        print(f'{key}: {value:.3f}')


=== Brazil ===
games_played: 10.000
win_rate: 0.500
draw_rate: 0.200
avg_goals_scored: 1.600
avg_goals_conceded: 1.000

=== Norway ===
games_played: 10.000
win_rate: 0.900
draw_rate: 0.000
avg_goals_scored: 3.800
avg_goals_conceded: 1.100


In [41]:
## 🔍 Prediction Validation - Brazil vs Norway

**Model prediction:** Loss (53%) — Norway wins
**Issue:** Norway's 90% win rate and 3.8 goals/game 
from UEFA qualifying against weak opposition
inflates their perceived strength

**Brazil's 50% win rate** from tough CONMEBOL 
qualifying is actually stronger than it appears

### Confirms Key Limitation
Quality of opposition is the biggest weakness 
in our current model. Adding ELO ratings would 
fix this — ELO accounts for opponent strength 
when updating team ratings.

SyntaxError: invalid character '—' (U+2014) (1011966146.py, line 3)

In [42]:
# ELO Rating System
# Starting ELO for all teams - 1500 is the standard default
STARTING_ELO = 1500
K_FACTOR = 32  # How much each match affects the rating

def expected_score(rating_a, rating_b):
    """
    Calculate expected score for team A against team B
    Returns probability of team A winning (0 to 1)
    """
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

# Test it
brazil_elo = 2000
norway_elo = 1600

expected = expected_score(brazil_elo, norway_elo)
print(f'Brazil ({brazil_elo}) vs Norway ({norway_elo})')
print(f'Expected score for Brazil: {expected:.3f}')
print(f'This means Brazil has a {expected*100:.1f}% chance of winning')

Brazil (2000) vs Norway (1600)
Expected score for Brazil: 0.909
This means Brazil has a 90.9% chance of winning


In [43]:
def calculate_elo_ratings(df, k=32, starting_elo=1500):
    """
    Calculate ELO ratings for all teams by processing
    every match chronologically.
    
    Key principle: only use past matches to calculate 
    ratings - no data leakage
    """
    elo_ratings = {}  # Stores current ELO for each team
    elo_history = []  # Stores ELO before each match
    
    for _, match in df.iterrows():
        home = match['home_team']
        away = match['away_team']
        
        # Assign starting ELO if team hasn't played before
        if home not in elo_ratings:
            elo_ratings[home] = starting_elo
        if away not in elo_ratings:
            elo_ratings[away] = starting_elo
        
        # Get current ratings BEFORE the match
        home_elo = elo_ratings[home]
        away_elo = elo_ratings[away]
        
        # Store pre-match ratings
        elo_history.append({
            'date': match['date'],
            'home_team': home,
            'away_team': away,
            'home_elo': home_elo,
            'away_elo': away_elo
        })
        
        # Calculate expected scores
        home_expected = expected_score(home_elo, away_elo)
        away_expected = expected_score(away_elo, home_elo)
        
        # Actual scores (1=win, 0.5=draw, 0=loss)
        if match['outcome'] == 'W':
            home_actual, away_actual = 1, 0
        elif match['outcome'] == 'D':
            home_actual, away_actual = 0.5, 0.5
        else:
            home_actual, away_actual = 0, 1
        
        # Update ELO ratings
        elo_ratings[home] = home_elo + k * (home_actual - home_expected)
        elo_ratings[away] = away_elo + k * (away_actual - away_expected)
    
    return pd.DataFrame(elo_history), elo_ratings

print('✅ ELO function defined!')

✅ ELO function defined!


In [44]:
# Calculate ELO ratings for all matches
print('Calculating ELO ratings for all 19,872 matches...')
print('Please wait...\n')

elo_history_df, final_elo = calculate_elo_ratings(model_df)

print(f'✅ ELO calculated!')
print(f'\nTop 10 teams by final ELO rating:')
top_teams = sorted(final_elo.items(), key=lambda x: x[1], reverse=True)[:10]
for rank, (team, elo) in enumerate(top_teams, 1):
    print(f'{rank}. {team:<25} ELO: {elo:.0f}')

Calculating ELO ratings for all 19,872 matches...
Please wait...

✅ ELO calculated!

Top 10 teams by final ELO rating:
1. Spain                     ELO: 1978
2. France                    ELO: 1942
3. Argentina                 ELO: 1934
4. Mexico                    ELO: 1909
5. England                   ELO: 1900
6. Morocco                   ELO: 1886
7. Japan                     ELO: 1862
8. Portugal                  ELO: 1856
9. Australia                 ELO: 1853
10. Brazil                    ELO: 1852


In [45]:
# Check Norway's ELO
print(f"Norway ELO: {final_elo.get('Norway', 1500):.0f}")
print(f"Brazil ELO: {final_elo.get('Brazil', 1500):.0f}")
print(f"Argentina ELO: {final_elo.get('Argentina', 1500):.0f}")


Norway ELO: 1768
Brazil ELO: 1852
Argentina ELO: 1934


In [46]:
# Merge ELO ratings into our model dataframe
model_df_elo = model_df.merge(
    elo_history_df[['date', 'home_team', 'away_team', 
                    'home_elo', 'away_elo']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

# Add ELO difference as a feature (positive = home team stronger)
model_df_elo['elo_difference'] = (
    model_df_elo['home_elo'] - model_df_elo['away_elo']
)

print(f'Shape: {model_df_elo.shape}')
print(f'\nNew ELO columns added:')
print(model_df_elo[['home_team', 'away_team', 
                     'home_elo', 'away_elo', 
                     'elo_difference']].head(5))

Shape: (19872, 19)

New ELO columns added:
       home_team   away_team  home_elo  away_elo  elo_difference
0       Thailand       Kenya    1500.0    1500.0             0.0
1       Colombia     Uruguay    1500.0    1500.0             0.0
2      Indonesia       Kenya    1500.0    1484.0            16.0
3  United States  Costa Rica    1500.0    1500.0             0.0
4     Costa Rica     Uruguay    1516.0    1516.0             0.0


In [48]:
# Retrain with ELO features included
drop_cols = ['date', 'home_team', 'away_team', 'tournament', 'outcome']

X_elo = model_df_elo.drop(columns=drop_cols)
y_elo = model_df_elo['outcome']

# Same time based split
train_mask = model_df_elo['date'] < pd.Timestamp('2022-01-01')
test_mask = model_df_elo['date'] >= pd.Timestamp('2022-01-01')

X_train_elo = X_elo[train_mask]
X_test_elo = X_elo[test_mask]
y_train_elo = y_elo[train_mask]
y_test_elo = y_elo[test_mask]

# Encode target
y_train_elo_enc = le.fit_transform(y_train_elo)
y_test_elo_enc = le.transform(y_test_elo)

# Train Gradient Boosting with ELO
gb_elo = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb_elo.fit(X_train_elo, y_train_elo_enc)
preds_elo = gb_elo.predict(X_test_elo)

accuracy_elo = accuracy_score(y_test_elo_enc, preds_elo)
print(f'Previous accuracy (no ELO): 56.08%')
print(f'New accuracy (with ELO):    {accuracy_elo*100:.2f}%')
print(f'\nImprovement: {(accuracy_elo*100 - 56.08):.2f}%')

Previous accuracy (no ELO): 56.08%
New accuracy (with ELO):    58.97%

Improvement: 2.89%


In [49]:
print('=== With ELO - Detailed Report ===\n')
print(classification_report(
    y_test_elo_enc,
    preds_elo,
    target_names=['Draw', 'Loss', 'Win']
))

=== With ELO - Detailed Report ===

              precision    recall  f1-score   support

        Draw       0.19      0.02      0.04       736
        Loss       0.56      0.63      0.60      1015
         Win       0.62      0.83      0.71      1537

    accuracy                           0.59      3288
   macro avg       0.46      0.50      0.45      3288
weighted avg       0.51      0.59      0.53      3288



In [50]:
## 📈 ELO Rating Improvement

Adding ELO ratings improved accuracy from 56.08% to 58.97% (+2.89%)

| Metric | Without ELO | With ELO |
|---|---|---|
| Accuracy | 56.08% | 58.97% |
| Win F1 | 0.68 | 0.71 |
| Loss F1 | 0.54 | 0.60 |
| Draw F1 | 0.02 | 0.04 |

### Key Insight
ELO effectively identifies stronger vs weaker teams,
improving Win and Loss prediction. Draws remain 
difficult — they are influenced by random match 
events beyond what historical stats can capture.

SyntaxError: invalid character '—' (U+2014) (2959687189.py, line 15)

In [51]:
def predict_match_elo(home_team, away_team, date, df, 
                      model, le, elo_ratings, neutral=True):
    match_date = pd.Timestamp(date)
    
    # Get team stats
    home_stats = get_team_stats(home_team, match_date, df)
    away_stats = get_team_stats(away_team, match_date, df)
    
    # Get ELO ratings
    home_elo = elo_ratings.get(home_team, 1500)
    away_elo = elo_ratings.get(away_team, 1500)
    elo_diff = home_elo - away_elo
    
    features = {
        'neutral': 1 if neutral else 0,
        'home_games_played': home_stats['games_played'],
        'home_win_rate': home_stats['win_rate'],
        'home_draw_rate': home_stats['draw_rate'],
        'home_avg_goals_scored': home_stats['avg_goals_scored'],
        'home_avg_goals_conceded': home_stats['avg_goals_conceded'],
        'away_games_played': away_stats['games_played'],
        'away_win_rate': away_stats['win_rate'],
        'away_draw_rate': away_stats['draw_rate'],
        'away_avg_goals_scored': away_stats['avg_goals_scored'],
        'away_avg_goals_conceded': away_stats['avg_goals_conceded'],
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_difference': elo_diff
    }
    
    feature_df = pd.DataFrame([features])
    prediction = model.predict(feature_df)
    probability = model.predict_proba(feature_df)
    
    outcome = le.inverse_transform(prediction)[0]
    probs = dict(zip(le.classes_, probability[0].round(3)))
    
    return outcome, probs, home_elo, away_elo

# Argentina vs Cape Verde with ELO
outcome, probs, h_elo, a_elo = predict_match_elo(
    'Argentina', 'Cape Verde', '2026-07-03',
    filtered, gb_elo, le, final_elo
)

print(f'=== Argentina vs Cape Verde (with ELO) ===')
print(f'Argentina ELO: {h_elo:.0f} | Cape Verde ELO: {a_elo:.0f}')
print(f'ELO difference: {h_elo-a_elo:.0f} (Argentina stronger)')
print(f'\nPredicted outcome: {outcome}')
print(f'\nProbabilities:')
for result, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 30)
    print(f'{result}: {bar} {prob*100:.1f}%')

=== Argentina vs Cape Verde (with ELO) ===
Argentina ELO: 1934 | Cape Verde ELO: 1619
ELO difference: 315 (Argentina stronger)

Predicted outcome: D

Probabilities:
D: ███████████████ 53.1%
W: ███████████ 39.5%
L: ██ 7.4%


In [52]:
## ⚠️ Persistent Limitation - Context of Statistics

Despite ELO improvement, Cape Verde's 70% draw rate
still dominates predictions against Argentina.

**Root cause:** Model cannot distinguish WHO draws
were against. Cape Verde's draws vs weak African 
nations are not comparable to facing Argentina.

**Fix requires:**
- Opponent-weighted statistics
- Head to head records
- Larger ELO difference thresholds for mismatches

**Key lesson:** Feature quality matters more than 
model complexity. Better data > better algorithm.

SyntaxError: unterminated string literal (detected at line 3) (898009103.py, line 3)

In [53]:
outcome, probs, h_elo, a_elo = predict_match_elo(
    'Brazil', 'Norway', '2026-07-05',
    filtered, gb_elo, le, final_elo
)

print(f'=== Brazil vs Norway (with ELO) ===')
print(f'Brazil ELO: {h_elo:.0f} | Norway ELO: {a_elo:.0f}')
print(f'ELO difference: {h_elo-a_elo:.0f}')
print(f'\nPredicted outcome: {outcome}')
print(f'\nProbabilities:')
for result, prob in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 30)
    print(f'{result}: {bar} {prob*100:.1f}%')

=== Brazil vs Norway (with ELO) ===
Brazil ELO: 1852 | Norway ELO: 1768
ELO difference: 84

Predicted outcome: L

Probabilities:
L: ███████████████ 50.2%
W: ███████████ 38.0%
D: ███ 11.8%


In [54]:
print(f'Expected score for Brazil: {expected_score(1852, 1768):.3f}')

Expected score for Brazil: 0.619


In [55]:
# Export data for Power BI
# We'll export 3 files - each serves a different purpose

# 1. Main match results with outcomes
model_df_elo.to_csv('powerbi_matches.csv', index=False)

# 2. Final ELO ratings for all teams
elo_df = pd.DataFrame([
    {'team': team, 'elo_rating': rating}
    for team, rating in final_elo.items()
]).sort_values('elo_rating', ascending=False)

elo_df.to_csv('powerbi_elo_ratings.csv', index=False)

# 3. Outcome summary by team
team_summary = []
for team in filtered['home_team'].unique():
    matches = filtered[
        (filtered['home_team'] == team) | 
        (filtered['away_team'] == team)
    ]
    total = len(matches)
    if total < 10:
        continue
    team_summary.append({
        'team': team,
        'total_matches': total,
        'elo_rating': final_elo.get(team, 1500)
    })

summary_df = pd.DataFrame(team_summary).sort_values(
    'elo_rating', ascending=False
)
summary_df.to_csv('powerbi_team_summary.csv', index=False)

print('✅ Files exported!')
print(f'powerbi_matches.csv — {len(model_df_elo)} rows')
print(f'powerbi_elo_ratings.csv — {len(elo_df)} teams')
print(f'powerbi_team_summary.csv — {len(summary_df)} teams')

✅ Files exported!
powerbi_matches.csv — 19872 rows
powerbi_elo_ratings.csv — 296 teams
powerbi_team_summary.csv — 246 teams
